# 02. Data Cleaning

**Objective:** clean the source datasets according to the rules defined after the data audit and documented in [`Cleaning rules`](../docs/cleaning_rules.pdf).

At this stage:
- raw files remain unchanged;
- all transformations are performed in Python;
- cleaned datasets are saved to the `data/processed` folder;
- CRM identifiers are processed as strings because they are technical keys, not numeric metrics.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.2f}'.format)


# Data Loading


In [2]:
# Project root directory
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DATA_PATH = PROJECT_ROOT / 'data' / 'processed'
NOTEBOOKS_PATH = PROJECT_ROOT / 'notebooks'

sys.path.append(str(NOTEBOOKS_PATH))


In [3]:
from project_util import (
    clean_id_column,
    clean_text_column,
    clean_money_column,
    standardize_deutsch_level,
    cleaning_summary,
    dict_to_check_df
)


In [4]:
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)


In [5]:
spend_raw = pd.read_excel(RAW_DATA_PATH / 'spend_raw.xlsx')

contacts_raw = pd.read_excel(
    RAW_DATA_PATH / 'contacts_raw.xlsx',
    dtype={'Id': 'string'}
)

calls_raw = pd.read_excel(
    RAW_DATA_PATH / 'calls_raw.xlsx',
    dtype={
        'Id': 'string',
        'CONTACTID': 'string'
    }
)

deals_raw = pd.read_excel(
    RAW_DATA_PATH / 'deals_raw.xlsx',
    dtype={
        'Id': 'string',
        'Contact Name': 'string'
    }
)


In [6]:
datasets = {
    'Spend': spend_raw,
    'Contacts': contacts_raw,
    'Calls': calls_raw,
    'Deals': deals_raw
}

load_check = []

for name, df in datasets.items():
    load_check.append({
        'Dataset': name,
        'Rows': df.shape[0],
        'Columns': df.shape[1]
    })

pd.DataFrame(load_check)


,Dataset,Rows,Columns
0,Spend,20779,8
1,Contacts,18548,4
2,Calls,95874,11
3,Deals,21595,23


# Cleaning the `Spend` Table

The `Spend` table is used to analyze marketing spend, impressions, clicks and advertising source performance.

Cleaning steps:
- convert `Date` to datetime;
- convert numeric fields to numeric format;
- clean text advertising fields;
- create a technical `Campaign` value for rows with spend and missing campaign;
- create the `Invalid CTR` flag for rows where `Clicks > Impressions`;
- keep rows with invalid CTR in the dataset, but exclude them from CTR calculations.


In [7]:
spend_clean = spend_raw.copy()


In [8]:
# Convert date to datetime format

spend_clean['Date'] = pd.to_datetime(
    spend_clean['Date'],
    errors='coerce'
)


In [9]:
# Convert numeric fields to numeric format

numeric_cols_spend = ['Impressions', 'Spend', 'Clicks']

for col in numeric_cols_spend:
    spend_clean[col] = pd.to_numeric(
        spend_clean[col],
        errors='coerce'
    )


In [10]:
# Clean text advertising fields

text_cols_spend = ['Source', 'Campaign', 'AdGroup', 'Ad']

for col in text_cols_spend:
    spend_clean[col] = clean_text_column(spend_clean[col])


In [11]:
# Replace missing values in key numeric fields with 0

spend_clean[['Impressions', 'Spend', 'Clicks']] = spend_clean[
    ['Impressions', 'Spend', 'Clicks']
].fillna(0)


In [12]:
# Create a technical campaign name for rows with spend and missing Campaign

missing_campaign_with_spend = (
    spend_clean['Campaign'].isna() &
    (spend_clean['Spend'] > 0)
)

spend_clean.loc[missing_campaign_with_spend, 'Campaign'] = (
    spend_clean.loc[missing_campaign_with_spend, 'Date'].dt.strftime('%Y-%m-%d') +
    '_' +
    spend_clean.loc[missing_campaign_with_spend, 'Source'].astype('string')
)


In [13]:
# Flag rows where CTR should not be calculated

spend_clean['Invalid CTR'] = spend_clean['Clicks'] > spend_clean['Impressions']


In [14]:
 # Calculate CTR only for rows with a valid Clicks-to-Impressions relationship

spend_clean['CTR'] = np.where(
    (spend_clean['Impressions'] > 0) & (~spend_clean['Invalid CTR']),
    spend_clean['Clicks'] / spend_clean['Impressions'],
    np.nan
)

# CPC can be calculated when clicks are greater than 0

spend_clean['CPC'] = np.where(
    spend_clean['Clicks'] > 0,
    spend_clean['Spend'] / spend_clean['Clicks'],
    np.nan
)


In [15]:
cleaning_summary(spend_raw, spend_clean, 'Spend')


Dataset summary: Spend


,Metric,Value
0,Rows before cleaning,20779
1,Rows after cleaning,20779
2,Columns before cleaning,8
3,Columns after cleaning,11


In [16]:
spend_raw


,Date,Source,Campaign,Impressions,Spend,Clicks,AdGroup,Ad
0,2023-07-03,Google Ads,gen_analyst_DE,6,0.00,0,NaN,NaN
1,2023-07-03,Google Ads,performancemax_eng_DE,4,0.01,1,NaN,NaN
2,2023-07-03,Facebook Ads,NaN,0,0.00,0,NaN,NaN
3,2023-07-03,Google Ads,NaN,0,0.00,0,NaN,NaN
4,2023-07-03,CRM,NaN,0,0.00,0,NaN,NaN
...,...,...,...,...,...,...,...,...
20774,2024-06-21,Facebook Ads,17.03.24wide_AT,7,0.07,0,wide,bloggersvideo16com_at
20775,2024-06-21,Tiktok Ads,12.07.2023wide_DE,61,0.16,0,wide,bloggersvideo14com
20776,2024-06-21,Partnership,NaN,0,0.00,0,NaN,NaN
20777,2024-06-21,Tiktok Ads,NaN,0,0.00,0,NaN,NaN


In [17]:
spend_clean


,Date,Source,Campaign,Impressions,Spend,Clicks,AdGroup,Ad,Invalid CTR,CTR,CPC
0,2023-07-03,Google Ads,gen_analyst_DE,6,0.00,0,<NA>,<NA>,False,0.00,NaN
1,2023-07-03,Google Ads,performancemax_eng_DE,4,0.01,1,<NA>,<NA>,False,0.25,0.01
2,2023-07-03,Facebook Ads,<NA>,0,0.00,0,<NA>,<NA>,False,NaN,NaN
3,2023-07-03,Google Ads,<NA>,0,0.00,0,<NA>,<NA>,False,NaN,NaN
4,2023-07-03,CRM,<NA>,0,0.00,0,<NA>,<NA>,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
20774,2024-06-21,Facebook Ads,17.03.24wide_AT,7,0.07,0,wide,bloggersvideo16com_at,False,0.00,NaN
20775,2024-06-21,Tiktok Ads,12.07.2023wide_DE,61,0.16,0,wide,bloggersvideo14com,False,0.00,NaN
20776,2024-06-21,Partnership,<NA>,0,0.00,0,<NA>,<NA>,False,NaN,NaN
20777,2024-06-21,Tiktok Ads,<NA>,0,0.00,0,<NA>,<NA>,False,NaN,NaN


In [18]:
spend_clean_checks = {
    'Missing Date values': spend_clean['Date'].isna().sum(),
    'Missing Source values': spend_clean['Source'].isna().sum(),
    'Missing Campaign values when Spend > 0': (
        spend_clean['Campaign'].isna() &
        (spend_clean['Spend'] > 0)
    ).sum(),
    'Missing Impressions values': spend_clean['Impressions'].isna().sum(),
    'Missing Spend values': spend_clean['Spend'].isna().sum(),
    'Missing Clicks values': spend_clean['Clicks'].isna().sum(),
    'Rows with Invalid CTR': spend_clean['Invalid CTR'].sum(),
    'Missing CTR values': spend_clean['CTR'].isna().sum(),
    'Missing CPC values': spend_clean['CPC'].isna().sum()
}

pd.DataFrame(
    spend_clean_checks.items(),
    columns=['Check', 'Result']
)


,Check,Result
0,Missing Date values,0
1,Missing Source values,0
2,Missing Campaign values when Spend > 0,0
3,Missing Impressions values,0
4,Missing Spend values,0
5,Missing Clicks values,0
6,Rows with Invalid CTR,1370
7,Missing CTR values,5751
8,Missing CPC values,8801


## `Spend` Cleaning Summary

During the cleaning stage, date, numeric and text fields in the `Spend` table were converted to the required formats.

For rows where `Campaign` was missing but advertising spend was recorded, a technical campaign value was created using the `Date_Source` rule.

Rows where `Clicks > Impressions` were flagged as `Invalid CTR`. These rows remain in the dataset and can be used for spend, clicks and CPC analysis, but they should not be used for CTR calculation.

The derived metrics `CTR` and `CPC` were also created for further marketing analysis.


# Cleaning the `Contacts` Table

The `Contacts` table is used as a CRM contact reference table and as a link to `Deals` and `Calls`.

Cleaning steps:
- convert `Id` to string format;
- clean `Contact Owner Name`;
- convert `Created Time` and `Modified Time` to datetime;
- create a logical date error flag when `Modified Time` is earlier than `Created Time`.


In [19]:
contacts_clean = contacts_raw.copy()


In [20]:
# Convert Id to string format

contacts_clean['Id'] = clean_id_column(contacts_clean['Id'])


In [21]:
# Clean the contact owner text field

contacts_clean['Contact Owner Name'] = clean_text_column(
    contacts_clean['Contact Owner Name']
)


In [22]:
# Convert date fields to datetime format

contacts_clean['Created Time'] = pd.to_datetime(
    contacts_clean['Created Time'],
    errors='coerce',
    dayfirst=True
)

contacts_clean['Modified Time'] = pd.to_datetime(
    contacts_clean['Modified Time'],
    errors='coerce',
    dayfirst=True
)


In [23]:
# Flag logical date errors

contacts_clean['Invalid Contact Dates'] = (
    contacts_clean['Created Time'].notna() &
    contacts_clean['Modified Time'].notna() &
    (contacts_clean['Modified Time'] < contacts_clean['Created Time'])
)


In [24]:
cleaning_summary(contacts_raw, contacts_clean, 'Contacts')


Dataset summary: Contacts


,Metric,Value
0,Rows before cleaning,18548
1,Rows after cleaning,18548
2,Columns before cleaning,4
3,Columns after cleaning,5


In [25]:
contacts_raw


,Id,Contact Owner Name,Created Time,Modified Time
0,5805028000000645014,Rachel White,27.06.2023 11:28,22.12.2023 13:34
1,5805028000000872003,Charlie Davis,03.07.2023 11:31,21.05.2024 10:23
2,5805028000000889001,Bob Brown,02.07.2023 22:37,21.12.2023 13:17
3,5805028000000907006,Bob Brown,03.07.2023 05:44,29.12.2023 15:20
4,5805028000000939010,Nina Scott,04.07.2023 10:11,16.04.2024 16:14
...,...,...,...,...
18543,5805028000056889209,Ulysses Adams,21.06.2024 12:11,21.06.2024 14:11
18544,5805028000056889351,Eva Kent,21.06.2024 13:32,21.06.2024 15:32
18545,5805028000056892018,Eva Kent,21.06.2024 10:21,21.06.2024 12:21
18546,5805028000056892055,Yara Edwards,21.06.2024 10:22,21.06.2024 12:23


In [26]:
contacts_clean


,Id,Contact Owner Name,Created Time,Modified Time,Invalid Contact Dates
0,5805028000000645014,Rachel White,2023-06-27 11:28:00,2023-12-22 13:34:00,False
1,5805028000000872003,Charlie Davis,2023-07-03 11:31:00,2024-05-21 10:23:00,False
2,5805028000000889001,Bob Brown,2023-07-02 22:37:00,2023-12-21 13:17:00,False
3,5805028000000907006,Bob Brown,2023-07-03 05:44:00,2023-12-29 15:20:00,False
4,5805028000000939010,Nina Scott,2023-07-04 10:11:00,2024-04-16 16:14:00,False
...,...,...,...,...,...
18543,5805028000056889209,Ulysses Adams,2024-06-21 12:11:00,2024-06-21 14:11:00,False
18544,5805028000056889351,Eva Kent,2024-06-21 13:32:00,2024-06-21 15:32:00,False
18545,5805028000056892018,Eva Kent,2024-06-21 10:21:00,2024-06-21 12:21:00,False
18546,5805028000056892055,Yara Edwards,2024-06-21 10:22:00,2024-06-21 12:23:00,False


In [27]:
contacts_clean_checks = {
    'Missing Id values': contacts_clean['Id'].isna().sum(),
    'Duplicate Id values': contacts_clean['Id'].dropna().duplicated().sum(),
    'Missing Contact Owner Name values': contacts_clean['Contact Owner Name'].isna().sum(),
    'Missing Created Time values': contacts_clean['Created Time'].isna().sum(),
    'Missing Modified Time values': contacts_clean['Modified Time'].isna().sum(),
    'Rows with Invalid Contact Dates': contacts_clean['Invalid Contact Dates'].sum()
}

pd.DataFrame(
    contacts_clean_checks.items(),
    columns=['Check', 'Result']
)


,Check,Result
0,Missing Id values,0
1,Duplicate Id values,0
2,Missing Contact Owner Name values,0
3,Missing Created Time values,0
4,Missing Modified Time values,0
5,Rows with Invalid Contact Dates,0


## `Contacts` Cleaning Summary

In the `Contacts` table, `Id` was converted to string format because CRM identifiers are technical keys, not numeric metrics.

`Created Time` and `Modified Time` were converted to datetime format.

The `Invalid Contact Dates` flag was created to identify rows where the contact modification date is earlier than the contact creation date.


# Cleaning the `Calls` Table

The `Calls` table is used to analyze sales team calling activity and to link calls to contacts.

Cleaning steps:
- convert `Id` and `CONTACTID` to string format;
- convert `Call Start Time` to datetime;
- convert `Call Duration (in seconds)` to numeric format;
- clean categorical text fields;
- remove `Dialled Number` and `Tag` because they are fully empty;
- check that, after processing `CONTACTID` as a string, calls can be correctly matched with the `Contacts` table.


In [28]:
calls_clean = calls_raw.copy()


In [29]:
# Convert CRM identifiers to string format

calls_clean['Id'] = clean_id_column(calls_clean['Id'])
calls_clean['CONTACTID'] = clean_id_column(calls_clean['CONTACTID'])


In [30]:
# Convert call start time to datetime format

calls_clean['Call Start Time'] = pd.to_datetime(
    calls_clean['Call Start Time'],
    errors='coerce',
    dayfirst=True
)


In [31]:
# Convert call duration to numeric format

calls_clean['Call Duration (in seconds)'] = pd.to_numeric(
    calls_clean['Call Duration (in seconds)'],
    errors='coerce'
)


In [32]:
# Clean categorical text fields

text_cols_calls = [
    'Call Owner Name',
    'Call Type',
    'Call Status',
    'Outgoing Call Status'
]

for col in text_cols_calls:
    calls_clean[col] = clean_text_column(calls_clean[col])


In [33]:
# Flag rows that should not be used in call duration analysis

calls_clean['Invalid Call Duration'] = (
    calls_clean['Call Duration (in seconds)'].isna() |
    (calls_clean['Call Duration (in seconds)'] < 0)
)


In [34]:
# Remove fully empty fields that are not used in analysis

calls_clean = calls_clean.drop(columns=['Dialled Number', 'Tag'])


In [35]:
cleaning_summary(calls_raw, calls_clean, 'Calls')


Dataset summary: Calls


,Metric,Value
0,Rows before cleaning,95874
1,Rows after cleaning,95874
2,Columns before cleaning,11
3,Columns after cleaning,10


In [36]:
calls_raw


,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Dialled Number,Outgoing Call Status,Scheduled in CRM,Tag
0,5805028000000805001,30.06.2023 08:43,John Doe,<NA>,Inbound,171.00,Received,NaN,NaN,NaN,NaN
1,5805028000000768006,30.06.2023 08:46,John Doe,<NA>,Outbound,28.00,Attended Dialled,NaN,Completed,0.00,NaN
2,5805028000000764027,30.06.2023 08:59,John Doe,<NA>,Outbound,24.00,Attended Dialled,NaN,Completed,0.00,NaN
3,5805028000000787003,30.06.2023 09:20,John Doe,5805028000000645014,Outbound,6.00,Attended Dialled,NaN,Completed,0.00,NaN
4,5805028000000768019,30.06.2023 09:30,John Doe,5805028000000645014,Outbound,11.00,Attended Dialled,NaN,Completed,0.00,NaN
...,...,...,...,...,...,...,...,...,...,...,...
95869,5805028000056889515,21.06.2024 15:30,Ulysses Adams,5805028000056564231,Outbound,6.00,Attended Dialled,NaN,Completed,0.00,NaN
95870,5805028000056875317,21.06.2024 15:30,Victor Barnes,5805028000054867023,Outbound,8.00,Attended Dialled,NaN,Completed,0.00,NaN
95871,5805028000056832495,21.06.2024 15:30,Kevin Parker,5805028000010617278,Outbound,5.00,Attended Dialled,NaN,Completed,0.00,NaN
95872,5805028000056893619,21.06.2024 15:30,Victor Barnes,5805028000056839048,Outbound,0.00,Unattended Dialled,NaN,Completed,0.00,NaN


In [37]:
calls_clean


,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Outgoing Call Status,Scheduled in CRM,Invalid Call Duration
0,5805028000000805001,2023-06-30 08:43:00,John Doe,<NA>,Inbound,171.00,Received,<NA>,NaN,False
1,5805028000000768006,2023-06-30 08:46:00,John Doe,<NA>,Outbound,28.00,Attended Dialled,Completed,0.00,False
2,5805028000000764027,2023-06-30 08:59:00,John Doe,<NA>,Outbound,24.00,Attended Dialled,Completed,0.00,False
3,5805028000000787003,2023-06-30 09:20:00,John Doe,5805028000000645014,Outbound,6.00,Attended Dialled,Completed,0.00,False
4,5805028000000768019,2023-06-30 09:30:00,John Doe,5805028000000645014,Outbound,11.00,Attended Dialled,Completed,0.00,False
...,...,...,...,...,...,...,...,...,...,...
95869,5805028000056889515,2024-06-21 15:30:00,Ulysses Adams,5805028000056564231,Outbound,6.00,Attended Dialled,Completed,0.00,False
95870,5805028000056875317,2024-06-21 15:30:00,Victor Barnes,5805028000054867023,Outbound,8.00,Attended Dialled,Completed,0.00,False
95871,5805028000056832495,2024-06-21 15:30:00,Kevin Parker,5805028000010617278,Outbound,5.00,Attended Dialled,Completed,0.00,False
95872,5805028000056893619,2024-06-21 15:30:00,Victor Barnes,5805028000056839048,Outbound,0.00,Unattended Dialled,Completed,0.00,False


In [38]:
calls_clean_checks = {
    'Missing Id values': calls_clean['Id'].isna().sum(),
    'Duplicate Id values': calls_clean['Id'].dropna().duplicated().sum(),

    'Missing CONTACTID values': calls_clean['CONTACTID'].isna().sum(),

    'Missing Call Start Time values': calls_clean['Call Start Time'].isna().sum(),

    'Missing Call Owner Name values': calls_clean['Call Owner Name'].isna().sum(),
    'Missing Call Type values': calls_clean['Call Type'].isna().sum(),
    'Missing Call Status values': calls_clean['Call Status'].isna().sum(),

    'Missing Call Duration values': calls_clean['Call Duration (in seconds)'].isna().sum(),
    'Rows with Invalid Call Duration': calls_clean['Invalid Call Duration'].sum(),

    'Missing Outgoing Call Status values': calls_clean['Outgoing Call Status'].isna().sum(),
    'Missing Scheduled in CRM values': calls_clean['Scheduled in CRM'].isna().sum()
}

pd.DataFrame(
    calls_clean_checks.items(),
    columns=['Check', 'Result']
)


,Check,Result
0,Missing Id values,0
1,Duplicate Id values,0
2,Missing CONTACTID values,3933
3,Missing Call Start Time values,0
4,Missing Call Owner Name values,0
5,Missing Call Type values,0
6,Missing Call Status values,0
7,Missing Call Duration values,83
8,Rows with Invalid Call Duration,83
9,Missing Outgoing Call Status values,8999


## Checking the `Calls` and `Contacts` Relationship after Cleaning


In [39]:
contacts_ids = set(contacts_clean['Id'].dropna())

calls_with_contactid = calls_clean[
    calls_clean['CONTACTID'].notna()
].copy()

calls_contacts_match_check = {
    'Calls rows with CONTACTID': calls_with_contactid.shape[0],
    'Calls rows without CONTACTID': calls_clean['CONTACTID'].isna().sum(),
    'Calls rows with CONTACTID found in Contacts': calls_with_contactid['CONTACTID'].isin(contacts_ids).sum(),
    'Calls rows with CONTACTID not found in Contacts': (~calls_with_contactid['CONTACTID'].isin(contacts_ids)).sum()
}

pd.DataFrame(
    calls_contacts_match_check.items(),
    columns=['Check', 'Result']
)


,Check,Result
0,Calls rows with CONTACTID,91941
1,Calls rows without CONTACTID,3933
2,Calls rows with CONTACTID found in Contacts,91941
3,Calls rows with CONTACTID not found in Contacts,0


## `Calls` Cleaning Summary

In the `Calls` table, `Id` and `CONTACTID` were converted to string format because CRM identifiers are technical keys.

After converting `CONTACTID` to string format, the `Calls.CONTACTID → Contacts.Id` relationship works correctly: filled `CONTACTID` values can be matched with contacts from the `Contacts` table.

Rows without `CONTACTID` represent a data limitation: they can be used in the overall call activity analysis, but not in relationships with contacts and deals.

`Call Start Time` was converted to datetime, and `Call Duration (in seconds)` was converted to numeric format.

The `Invalid Call Duration` flag was created for rows with missing or invalid call duration. These rows remain in the dataset, but should not be used in call duration analysis.

`Dialled Number` and `Tag` were removed from the cleaned table because they are fully empty and do not provide analytical value in the current export.


# Cleaning the `Deals` Table

The `Deals` table is the main table of the project. It is used to analyze the funnel, payments, products, sources, managers and revenue.

Cleaning steps:
- convert CRM identifiers to string format;
- clean text fields;
- remove rows without key fields;
- convert dates to datetime;
- convert SLA to time format;
- clean financial fields;
- create technical features for further analysis.


In [40]:
deals_clean = deals_raw.copy()


In [41]:
# Convert CRM identifiers to string format

deals_clean['Id'] = clean_id_column(deals_clean['Id'])
deals_clean['Contact Name'] = clean_id_column(deals_clean['Contact Name'])


In [42]:
# Clean text fields

text_cols_deals = [
    'Deal Owner Name',
    'Quality',
    'Stage',
    'Lost Reason',
    'Page',
    'Campaign',
    'Content',
    'Term',
    'Source',
    'Payment Type',
    'Product',
    'Education Type',
    'City',
    'Level of Deutsch'
]

for col in text_cols_deals:
    deals_clean[col] = clean_text_column(deals_clean[col])


In [43]:
# Exclude empty or technical rows without key fields

key_fields = ['Id', 'Stage', 'Source', 'Created Time']

missing_key_fields = deals_clean[key_fields].isna().any(axis=1)

print('Rows without key fields:', missing_key_fields.sum())

deals_clean = deals_clean[~missing_key_fields].copy()


Rows without key fields: 2


In [44]:
# Convert date fields to datetime format

deals_clean['Created Time'] = pd.to_datetime(
    deals_clean['Created Time'],
    errors='coerce',
    dayfirst=True
)

deals_clean['Closing Date'] = pd.to_datetime(
    deals_clean['Closing Date'],
    errors='coerce',
    dayfirst=True
)


In [45]:
# Calendar dates without time component

deals_clean['Created Date'] = deals_clean['Created Time'].dt.normalize()
deals_clean['Closing Date Only'] = deals_clean['Closing Date'].dt.normalize()


In [46]:
# Flag rows where the closing date is earlier than the creation date

deals_clean['Invalid Deal Dates'] = (
    deals_clean['Created Date'].notna() &
    deals_clean['Closing Date Only'].notna() &
    (deals_clean['Closing Date Only'] < deals_clean['Created Date'])
)


In [47]:
# Deal duration in days

deals_clean['Deal Duration Days'] = np.where(
    (~deals_clean['Invalid Deal Dates']) &
    deals_clean['Created Date'].notna() &
    deals_clean['Closing Date Only'].notna(),
    (deals_clean['Closing Date Only'] - deals_clean['Created Date']).dt.days,
    np.nan
)


In [48]:
# Convert SLA to timedelta format

deals_clean['SLA'] = pd.to_timedelta(
    deals_clean['SLA'].astype(str),
    errors='coerce'
)


In [49]:
# Clean and standardize financial fields

deals_clean['Initial Amount Paid'] = clean_money_column(
    deals_clean['Initial Amount Paid']
)

deals_clean['Offer Total Amount'] = clean_money_column(
    deals_clean['Offer Total Amount']
)


In [50]:
# Flag rows where the first payment is greater than the total offer amount

deals_clean['Initial Amount Greater Than Offer'] = (
    deals_clean['Initial Amount Paid'].notna() &
    deals_clean['Offer Total Amount'].notna() &
    (deals_clean['Initial Amount Paid'] > deals_clean['Offer Total Amount'])
)


In [51]:
# Create the paid deal flag

deals_clean['Is Paid'] = deals_clean['Stage'] == 'Payment Done'


In [52]:
# Actual revenue is calculated only for paid deals

deals_clean['Revenue'] = np.where(
    deals_clean['Is Paid'],
    deals_clean['Initial Amount Paid'],
    np.nan
)


In [53]:
# Flag paid deals without recorded payment amount

deals_clean['Unknown Payment Amount'] = (
    deals_clean['Is Paid'] &
    deals_clean['Initial Amount Paid'].isna()
)


In [54]:
# Flag symbolic payments or demo-access payments

deals_clean['Is Symbolic Payment'] = (
    deals_clean['Is Paid'] &
    deals_clean['Initial Amount Paid'].notna() &
    (deals_clean['Initial Amount Paid'] <= 10)
)


### Cleaning Categorical Fields in `Deals`

At this step, categorical fields used in funnel, lead quality, product, payment type and education format analysis are processed.

Missing values are filled with `Unknown` only where this is needed for further analysis and grouping.


In [55]:
# Fill missing Quality values with Unknown

deals_clean['Quality'] = deals_clean['Quality'].fillna('Unknown')


In [56]:
# Fill missing values with Unknown
# for paid deals with missing Product

deals_clean.loc[
    deals_clean['Is Paid'] & deals_clean['Product'].isna(),
    'Product'
] = 'Unknown'

# for paid deals with missing Education Type

deals_clean.loc[
    deals_clean['Is Paid'] & deals_clean['Education Type'].isna(),
    'Education Type'
] = 'Unknown'

# for paid deals with missing Payment Type

deals_clean.loc[
    deals_clean['Is Paid'] & deals_clean['Payment Type'].isna(),
    'Payment Type'
] = 'Unknown'


## Cleaning the `City` Field

The `City` field is used for geographic analysis. Missing values, the `-` value and non-interpretable values are marked as `Unknown`.

Rows with `City = Unknown` remain in the overall analysis, but may be shown as a separate category in geographic analysis.


In [57]:
deals_clean['City'] = clean_text_column(deals_clean['City'])

unknown_city_values = ['-', '—', 'n/a', 'N/A', 'na', 'NA', 'None', 'none']

deals_clean['City'] = deals_clean['City'].replace(unknown_city_values, pd.NA)

deals_clean['City'] = deals_clean['City'].fillna('Unknown')


In [58]:
city_before_after = pd.DataFrame({
    'City before cleaning': deals_raw.loc[deals_clean.index, 'City'],
    'City after cleaning': deals_clean['City']
})

city_mapping_check = (
    city_before_after
    .drop_duplicates()
    .sort_values('City after cleaning')
)

city_mapping_check.head(50)


,City before cleaning,City after cleaning
1254,Aachen,Aachen
7532,Aalen,Aalen
9972,Abensberg,Abensberg
6488,Achberg,Achberg
12990,Adelebsen,Adelebsen
1959,Adelschlag,Adelschlag
4249,Ahad Al Masarihah,Ahad Al Masarihah
11432,Ahaus,Ahaus
17215,Ahrensburg,Ahrensburg
8547,Aichach,Aichach


In [59]:
city_unknown_examples = (
    city_before_after[
        city_before_after['City after cleaning'] == 'Unknown'
    ]
    .drop_duplicates()
)

city_unknown_examples.head(50)


,City before cleaning,City after cleaning
0,NaN,Unknown
243,-,Unknown


In [60]:
city_clean_check = pd.DataFrame({
    'Row Count': deals_clean['City'].value_counts(dropna=False),
    'Share, %': (deals_clean['City'].value_counts(normalize=True, dropna=False) * 100).round(2)
})

city_clean_check.head(30)


,Row Count,"Share, %"
City,,
Unknown,19430,89.98
Berlin,182,0.84
München,74,0.34
Hamburg,62,0.29
Nürnberg,45,0.21
Leipzig,45,0.21
Düsseldorf,33,0.15
Dresden,28,0.13
Frankfurt,27,0.13


## Cleaning the `Level of Deutsch` Field

The `Level of Deutsch` field is used to analyze how German language level is related to deal success.

Values are standardized to `A1`, `A2`, `B1`, `B2`, `C1`, `C2`. Invalid, free-text and missing values are converted to `Unknown`.


In [61]:
deals_clean['Level of Deutsch'] = deals_clean['Level of Deutsch'].apply(
    standardize_deutsch_level
)


In [62]:
# Compare Level of Deutsch values before and after cleaning

deutsch_before_after = pd.DataFrame({
    'Level of Deutsch before cleaning': deals_raw.loc[deals_clean.index, 'Level of Deutsch'],
    'Level of Deutsch after cleaning': deals_clean['Level of Deutsch']
})

deutsch_mapping_check = (
    deutsch_before_after
    .drop_duplicates()
    .sort_values('Level of Deutsch after cleaning')
)

deutsch_mapping_check.head(50)


,Level of Deutsch before cleaning,Level of Deutsch after cleaning
13236,A1-A2,A1
15323,a0-a1,A1
4708,A1,A1
12306,учит A2,A2
17715,a2-б1,A2
18528,a2 (b1 экзамен 15 июня),A2
13299,A2 (идет доучивать В1 - 300 часов; предположит...,A2
3774,a2,A2
71,A2,A2
15420,A2 (идет на В1),A2


In [63]:
deutsch_level_check = pd.DataFrame({
    'Row Count': deals_clean['Level of Deutsch'].value_counts(dropna=False),
    'Share, %': (deals_clean['Level of Deutsch'].value_counts(normalize=True, dropna=False) * 100).round(2)
})

deutsch_level_check


,Row Count,"Share, %"
Level of Deutsch,,
Unknown,21096,97.70
B1,382,1.77
B2,76,0.35
A2,28,0.13
A1,6,0.03
C1,4,0.02
C2,1,0.00


## Cleaning `Course duration` and `Months of study`

`Course duration` and `Months of study` are used for student and education duration analysis.

Values are converted to numeric format. If `Months of study` is greater than `Course duration`, a logical error flag is created.


In [64]:
# Convert Course duration and Months of study to numeric format

deals_clean['Course duration'] = pd.to_numeric(
    deals_clean['Course duration'],
    errors='coerce'
)

deals_clean['Months of study'] = pd.to_numeric(
    deals_clean['Months of study'],
    errors='coerce'
)


In [65]:
# Flag negative values in course duration or months of study

deals_clean['Invalid Study Values'] = (
    (deals_clean['Course duration'].notna() & (deals_clean['Course duration'] < 0)) |
    (deals_clean['Months of study'].notna() & (deals_clean['Months of study'] < 0))
)


In [66]:
# Flag rows where Months of study is greater than Course duration

deals_clean['Months Greater Than Course Duration'] = (
    deals_clean['Course duration'].notna() &
    deals_clean['Months of study'].notna() &
    (deals_clean['Months of study'] > deals_clean['Course duration'])
)


## Checking the `Deals` and `Contacts` Relationship after Cleaning


In [67]:
contacts_ids = set(contacts_clean['Id'].dropna())

deals_with_contact = deals_clean[
    deals_clean['Contact Name'].notna()
].copy()

deals_contacts_match_check = {
    'Deals rows with Contact Name': deals_with_contact.shape[0],
    'Deals rows without Contact Name': deals_clean['Contact Name'].isna().sum(),
    'Deals rows with Contact Name found in Contacts': deals_with_contact['Contact Name'].isin(contacts_ids).sum(),
    'Deals rows with Contact Name not found in Contacts': (~deals_with_contact['Contact Name'].isin(contacts_ids)).sum()
}

pd.DataFrame(
    deals_contacts_match_check.items(),
    columns=['Check', 'Result']
)


,Check,Result
0,Deals rows with Contact Name,21532
1,Deals rows without Contact Name,61
2,Deals rows with Contact Name found in Contacts,21531
3,Deals rows with Contact Name not found in Cont...,1


In [68]:
cleaning_summary(deals_raw, deals_clean, 'Deals')


Dataset summary: Deals


,Metric,Value
0,Rows before cleaning,21595
1,Rows after cleaning,21593
2,Columns before cleaning,23
3,Columns after cleaning,34


In [69]:
deals_raw


,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,Term,Source,Payment Type,Product,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch
0,5805028000056864695,Ben Hall,NaN,NaN,New Lead,NaN,/eng/test,03.07.23women,NaN,v16,women,Facebook Ads,NaN,NaN,NaN,21.06.2024 15:30,NaN,NaN,NaN,NaN,5805028000056849495,NaN,NaN
1,5805028000056859489,Ulysses Adams,NaN,NaN,New Lead,NaN,/at-eng,NaN,NaN,NaN,NaN,Organic,NaN,Web Developer,Morning,21.06.2024 15:23,6.00,NaN,0,2000,5805028000056834471,NaN,NaN
2,5805028000056832357,Ulysses Adams,21.06.2024,D - Non Target,Lost,Non target,/at-eng,engwien_AT,00:26:43,b1-at,21_06_2024,Telegram posts,NaN,NaN,NaN,21.06.2024 14:45,NaN,NaN,NaN,NaN,5805028000056854421,NaN,NaN
3,5805028000056824246,Eva Kent,21.06.2024,E - Non Qualified,Lost,Invalid number,/eng,04.07.23recentlymoved_DE,01:00:04,bloggersvideo14com,recentlymoved,Facebook Ads,NaN,NaN,NaN,21.06.2024 13:32,NaN,NaN,NaN,NaN,5805028000056889351,NaN,NaN
4,5805028000056873292,Ben Hall,21.06.2024,D - Non Target,Lost,Non target,/eng,discovery_DE,00:53:12,website,NaN,Google Ads,NaN,NaN,NaN,21.06.2024 13:21,NaN,NaN,NaN,NaN,5805028000056876176,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21590,5805028000000945016,Jane Smith,29.08.2023,A - High,Lost,Changed Decision,eng/digital-marketing,02.07.23wide_DE,"56 days, 19:01:59",b3,wide,Facebook Ads,NaN,NaN,NaN,03.07.2023 20:39,NaN,NaN,NaN,NaN,5805028000000968001,NaN,NaN
21591,5805028000000927004,Bob Brown,09.07.2023,D - Non Target,Lost,Does not speak English,eng/digital-marketing,03.07.23women,NaN,b3,women,Facebook Ads,NaN,NaN,NaN,03.07.2023 20:17,NaN,NaN,NaN,NaN,5805028000000961001,NaN,NaN
21592,5805028000000922001,Bob Brown,03.07.2023,E - Non Qualified,Lost,Refugee,/,NaN,"4 days, 22:47:14",NaN,NaN,Organic,NaN,NaN,NaN,03.07.2023 17:03,NaN,NaN,0,0,5805028000001009140,NaN,NaN
21593,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN


In [70]:
deals_clean


,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,Term,Source,Payment Type,Product,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch,Created Date,Closing Date Only,Invalid Deal Dates,Deal Duration Days,Initial Amount Greater Than Offer,Is Paid,Revenue,Unknown Payment Amount,Is Symbolic Payment,Invalid Study Values,Months Greater Than Course Duration
0,5805028000056864695,Ben Hall,NaT,Unknown,New Lead,<NA>,/eng/test,03.07.23women,NaT,v16,women,Facebook Ads,<NA>,<NA>,<NA>,2024-06-21 15:30:00,NaN,NaN,NaN,NaN,5805028000056849495,Unknown,Unknown,2024-06-21,NaT,False,NaN,False,False,NaN,False,False,False,False
1,5805028000056859489,Ulysses Adams,NaT,Unknown,New Lead,<NA>,/at-eng,<NA>,NaT,<NA>,<NA>,Organic,<NA>,Web Developer,Morning,2024-06-21 15:23:00,6.00,NaN,0.00,"2,000.00",5805028000056834471,Unknown,Unknown,2024-06-21,NaT,False,NaN,False,False,NaN,False,False,False,False
2,5805028000056832357,Ulysses Adams,2024-06-21,D - Non Target,Lost,Non target,/at-eng,engwien_AT,0 days 00:26:43,b1-at,21_06_2024,Telegram posts,<NA>,<NA>,<NA>,2024-06-21 14:45:00,NaN,NaN,NaN,NaN,5805028000056854421,Unknown,Unknown,2024-06-21,2024-06-21,False,0.00,False,False,NaN,False,False,False,False
3,5805028000056824246,Eva Kent,2024-06-21,E - Non Qualified,Lost,Invalid number,/eng,04.07.23recentlymoved_DE,0 days 01:00:04,bloggersvideo14com,recentlymoved,Facebook Ads,<NA>,<NA>,<NA>,2024-06-21 13:32:00,NaN,NaN,NaN,NaN,5805028000056889351,Unknown,Unknown,2024-06-21,2024-06-21,False,0.00,False,False,NaN,False,False,False,False
4,5805028000056873292,Ben Hall,2024-06-21,D - Non Target,Lost,Non target,/eng,discovery_DE,0 days 00:53:12,website,<NA>,Google Ads,<NA>,<NA>,<NA>,2024-06-21 13:21:00,NaN,NaN,NaN,NaN,5805028000056876176,Unknown,Unknown,2024-06-21,2024-06-21,False,0.00,False,False,NaN,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21588,5805028000000970006,Jane Smith,2023-07-04,E - Non Qualified,Lost,Duplicate,eng/digital-marketing,03.07.23women,NaT,b3,women,Facebook Ads,<NA>,<NA>,<NA>,2023-07-04 07:10:00,NaN,NaN,NaN,NaN,5805028000000979006,Unknown,Unknown,2023-07-04,2023-07-04,False,0.00,False,False,NaN,False,False,False,False
21589,5805028000000948010,Jane Smith,2023-08-29,B - Medium,Lost,needs time to think,eng/digital-marketing,03.07.23women,NaT,b3,women,Facebook Ads,<NA>,<NA>,<NA>,2023-07-04 07:10:00,NaN,NaN,NaN,NaN,5805028000000979006,Unknown,Unknown,2023-07-04,2023-08-29,False,56.00,False,False,NaN,False,False,False,False
21590,5805028000000945016,Jane Smith,2023-08-29,A - High,Lost,Changed Decision,eng/digital-marketing,02.07.23wide_DE,56 days 19:01:59,b3,wide,Facebook Ads,<NA>,<NA>,<NA>,2023-07-03 20:39:00,NaN,NaN,NaN,NaN,5805028000000968001,Unknown,Unknown,2023-07-03,2023-08-29,False,57.00,False,False,NaN,False,False,False,False
21591,5805028000000927004,Bob Brown,2023-07-09,D - Non Target,Lost,Does not speak English,eng/digital-marketing,03.07.23women,NaT,b3,women,Facebook Ads,<NA>,<NA>,<NA>,2023-07-03 20:17:00,NaN,NaN,NaN,NaN,5805028000000961001,Unknown,Unknown,2023-07-03,2023-07-09,False,6.00,False,False,NaN,False,False,False,False


In [71]:
deals_clean_checks = {
    'Missing Id values': deals_clean['Id'].isna().sum(),
    'Duplicate Id values': deals_clean['Id'].dropna().duplicated().sum(),

    'Missing Stage values': deals_clean['Stage'].isna().sum(),
    'Missing Source values': deals_clean['Source'].isna().sum(),
    'Missing Created Time values': deals_clean['Created Time'].isna().sum(),

    'Paid deals: Payment Done': deals_clean['Is Paid'].sum(),
    'Payment Done without payment amount': deals_clean['Unknown Payment Amount'].sum(),
    'Symbolic payments': deals_clean['Is Symbolic Payment'].sum(),

    'Payment Done without Product': (
        deals_clean['Is Paid'] & deals_clean['Product'].isna()
    ).sum(),
    'Payment Done without Education Type': (
        deals_clean['Is Paid'] & deals_clean['Education Type'].isna()
    ).sum(),
    'Payment Done without Payment Type': (
        deals_clean['Is Paid'] & deals_clean['Payment Type'].isna()
    ).sum(),

    'Rows with Invalid Deal Dates': deals_clean['Invalid Deal Dates'].sum(),
    'Rows with Initial Amount Greater Than Offer': deals_clean['Initial Amount Greater Than Offer'].sum(),
    'Rows with Invalid Study Values': deals_clean['Invalid Study Values'].sum(),
    'Rows with Months Greater Than Course Duration': deals_clean['Months Greater Than Course Duration'].sum(),

    'Missing Quality values': deals_clean['Quality'].isna().sum(),
    'City = Unknown': (deals_clean['City'] == 'Unknown').sum(),
    'Level of Deutsch = Unknown': (deals_clean['Level of Deutsch'] == 'Unknown').sum()
}

pd.DataFrame(
    deals_clean_checks.items(),
    columns=['Check', 'Result']
)


,Check,Result
0,Missing Id values,0
1,Duplicate Id values,0
2,Missing Stage values,0
3,Missing Source values,0
4,Missing Created Time values,0
5,Paid deals: Payment Done,858
6,Payment Done without payment amount,15
7,Symbolic payments,4
8,Payment Done without Product,0
9,Payment Done without Education Type,0


## `Deals` Cleaning Summary

The `Deals` table was cleaned across key fields, dates, financial values, categorical attributes and technical identifiers.

CRM identifiers `Id` and `Contact Name` were converted to string format. Rows without key fields `Id`, `Stage`, `Source` and `Created Time` were excluded from the cleaned table.

`Created Time` and `Closing Date` were converted to datetime. Calendar dates without time were created for deal duration analysis. Rows where the closing date is earlier than the creation date were flagged as `Invalid Deal Dates`.

Financial fields `Initial Amount Paid` and `Offer Total Amount` were cleaned from currency symbols and converted to numeric format. The fields `Is Paid`, `Revenue`, `Unknown Payment Amount`, `Is Symbolic Payment` and `Initial Amount Greater Than Offer` were created.

For paid deals, missing values in `Product`, `Education Type` and `Payment Type` were filled as `Unknown` to preserve these deals in payment and conversion analysis.

`Quality` was filled as `Unknown` for deals without lead quality assessment. `City` and `Level of Deutsch` were also standardized with a separate `Unknown` category.

`Course duration` and `Months of study` were converted to numeric format. Additional data quality flags were created for logically invalid values.


# Post-Cleaning Validation

At this stage, a final summary check is performed for the cleaned datasets.

Detailed checks for missing values, duplicates and relationships between tables were already performed inside each cleaning block. This section records the overall cleaning result: changes in table size, created technical fields and key indicators used in the next analysis stages.


In [72]:
clean_datasets = {
    'Spend': (spend_raw, spend_clean),
    'Contacts': (contacts_raw, contacts_clean),
    'Calls': (calls_raw, calls_clean),
    'Deals': (deals_raw, deals_clean)
}

cleaning_result_summary = []

for name, (raw_df, clean_df) in clean_datasets.items():
    cleaning_result_summary.append({
        'Dataset': name,
        'Rows Before Cleaning': raw_df.shape[0],
        'Rows After Cleaning': clean_df.shape[0],
        'Row Change': clean_df.shape[0] - raw_df.shape[0],
        'Columns Before Cleaning': raw_df.shape[1],
        'Columns After Cleaning': clean_df.shape[1],
        'Column Change': clean_df.shape[1] - raw_df.shape[1]
    })

cleaning_result_summary_df = pd.DataFrame(cleaning_result_summary)

cleaning_result_summary_df


,Dataset,Rows Before Cleaning,Rows After Cleaning,Row Change,Columns Before Cleaning,Columns After Cleaning,Column Change
0,Spend,20779,20779,0,8,11,3
1,Contacts,18548,18548,0,4,5,1
2,Calls,95874,95874,0,11,10,-1
3,Deals,21595,21593,-2,23,34,11


In [73]:
# Check key ID fields

id_checks = {
    'Spend — rows': spend_clean.shape[0],

    'Contacts — missing Id values': contacts_clean['Id'].isna().sum(),
    'Contacts — duplicate Id values': contacts_clean['Id'].dropna().duplicated().sum(),

    'Calls — missing Id values': calls_clean['Id'].isna().sum(),
    'Calls — duplicate Id values': calls_clean['Id'].dropna().duplicated().sum(),
    'Calls — missing CONTACTID values': calls_clean['CONTACTID'].isna().sum(),

    'Deals — missing Id values': deals_clean['Id'].isna().sum(),
    'Deals — duplicate Id values': deals_clean['Id'].dropna().duplicated().sum(),
    'Deals — missing Contact Name values': deals_clean['Contact Name'].isna().sum()
}

pd.DataFrame(
    id_checks.items(),
    columns=['Check', 'Result']
)


,Check,Result
0,Spend — rows,20779
1,Contacts — missing Id values,0
2,Contacts — duplicate Id values,0
3,Calls — missing Id values,0
4,Calls — duplicate Id values,0
5,Calls — missing CONTACTID values,3933
6,Deals — missing Id values,0
7,Deals — duplicate Id values,0
8,Deals — missing Contact Name values,61


In [74]:
final_cleaning_metrics = {
    'Spend: rows with Invalid CTR': spend_clean['Invalid CTR'].sum(),
    'Spend: rows with calculated CTR': spend_clean['CTR'].notna().sum(),
    'Spend: rows with calculated CPC': spend_clean['CPC'].notna().sum(),

    'Calls: rows with Invalid Call Duration': calls_clean['Invalid Call Duration'].sum(),

    'Deals: total deals after cleaning': deals_clean.shape[0],
    'Deals: paid deals: Payment Done': deals_clean['Is Paid'].sum(),
    'Deals: Payment Done without payment amount': deals_clean['Unknown Payment Amount'].sum(),
    'Deals: symbolic payments': deals_clean['Is Symbolic Payment'].sum(),

    'Deals: rows with Invalid Deal Dates': deals_clean['Invalid Deal Dates'].sum(),
    'Deals: rows with Initial Amount Greater Than Offer': deals_clean['Initial Amount Greater Than Offer'].sum(),
    'Deals: rows with Invalid Study Values': deals_clean['Invalid Study Values'].sum(),
    'Deals: rows with Months Greater Than Course Duration': deals_clean['Months Greater Than Course Duration'].sum(),

    'Deals: Product = Unknown': (deals_clean['Product'] == 'Unknown').sum(),
    'Deals: Education Type = Unknown': (deals_clean['Education Type'] == 'Unknown').sum(),
    'Deals: Payment Type = Unknown': (deals_clean['Payment Type'] == 'Unknown').sum(),
    'Deals: Quality = Unknown': (deals_clean['Quality'] == 'Unknown').sum(),
    'Deals: City = Unknown': (deals_clean['City'] == 'Unknown').sum(),
    'Deals: Level of Deutsch = Unknown': (deals_clean['Level of Deutsch'] == 'Unknown').sum()
}

final_cleaning_metrics_df = pd.DataFrame(
    final_cleaning_metrics.items(),
    columns=['Metric', 'Value']
)

final_cleaning_metrics_df


,Metric,Value
0,Spend: rows with Invalid CTR,1370
1,Spend: rows with calculated CTR,15028
2,Spend: rows with calculated CPC,11978
3,Calls: rows with Invalid Call Duration,83
4,Deals: total deals after cleaning,21593
5,Deals: paid deals: Payment Done,858
6,Deals: Payment Done without payment amount,15
7,Deals: symbolic payments,4
8,Deals: rows with Invalid Deal Dates,44
9,Deals: rows with Initial Amount Greater Than O...,58


## Final Cleaning Validation Summary

After cleaning, all tables were converted to a format suitable for further analysis.

Detailed data quality checks were performed inside each table-specific cleaning block. The final summary confirms that the cleaned datasets are ready for the next project stages: EDA, business metrics, unit economics and dashboard development.


# Saving Cleaned Data

At this step, cleaned datasets are saved to the `data/processed` folder.

The cleaned files will be used in the following project stages:
- EDA;
- business metrics calculation;
- unit economics;
- dashboard development;
- report and presentation preparation.


In [75]:
# Save cleaned tables to CSV with Excel/Google Sheets-friendly encoding

spend_clean.to_csv(
    PROCESSED_DATA_PATH / 'spend_clean.csv',
    index=False,
    encoding='utf-8-sig'
)

contacts_clean.to_csv(
    PROCESSED_DATA_PATH / 'contacts_clean.csv',
    index=False,
    encoding='utf-8-sig'
)

calls_clean.to_csv(
    PROCESSED_DATA_PATH / 'calls_clean.csv',
    index=False,
    encoding='utf-8-sig'
)

deals_clean.to_csv(
    PROCESSED_DATA_PATH / 'deals_clean.csv',
    index=False,
    encoding='utf-8-sig'
)


In [76]:
# Save cleaned tables to Excel for visual review

spend_clean.to_excel(
    PROCESSED_DATA_PATH / 'spend_clean.xlsx',
    index=False
)

contacts_clean.to_excel(
    PROCESSED_DATA_PATH / 'contacts_clean.xlsx',
    index=False
)

calls_clean.to_excel(
    PROCESSED_DATA_PATH / 'calls_clean.xlsx',
    index=False
)

deals_clean.to_excel(
    PROCESSED_DATA_PATH / 'deals_clean.xlsx',
    index=False
)


## Data Cleaning Stage Summary

Cleaned versions of all four tables were saved to the `data/processed` folder:

- `spend_clean`;
- `contacts_clean`;
- `calls_clean`;
- `deals_clean`.

During the cleaning stage, the rules documented after the data audit and data quality assessment were applied:

- CRM identifiers were converted to string format;
- dates and numeric fields were converted to appropriate formats;
- financial fields were cleaned and normalized;
- data quality flags were created;
- rows with non-critical limitations were preserved, while flags were added to exclude them from specific calculations when needed;
- derived fields were prepared for further analysis, including `Is Paid`, `Revenue`, `CTR` and `CPC`.

The cleaned data is ready for the next project stage: EDA and business metrics calculation.
